# Piloto para escolha do dataset — Transfer Learning e Fine-Tuning em CNNs

**Disciplina:** Inteligência Computacional / Machine Learning — Profa. Polyana Santos Fonseca Nascimento (CESUPA)

**Objetivo deste notebook:** decidir, **com base em evidências**, qual destes três datasets usar no trabalho:

| # | Dataset | Fonte |
|---|---------|-------|
| 1 | **BRACOL** — folhas de café arábica, versão *symptom* | Krohling, Esgario & Ventura (2019), Mendeley Data, DOI [10.17632/yy2k5y8mxg.1](https://doi.org/10.17632/yy2k5y8mxg.1); artigo: Esgario et al. (2020), *Computers and Electronics in Agriculture* 169:105162 |
| 2 | **PBC** — células normais de sangue periférico | Acevedo et al. (2020), *Data in Brief* 30:105474; Mendeley Data, DOI [10.17632/snkd93bnjr.1](https://doi.org/10.17632/snkd93bnjr.1) |
| 3 | **DeepWeeds** — plantas daninhas da Austrália | Olsen et al. (2019), *Scientific Reports* 9:2058, DOI [10.1038/s41598-018-38343-3](https://doi.org/10.1038/s41598-018-38343-3); também em `tensorflow_datasets` (`deep_weeds`) |

**Como rodar:** Google Colab → *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4* → *Executar tudo*.
A célula de configuração pede acesso ao Google Drive (para salvar divisões, figuras e resultados entre sessões). Se não quiser, mude `USE_DRIVE = False`.

**Tempo aproximado (ordem de grandeza, T4):** BRACOL alguns minutos; PBC e DeepWeeds ~15–25 min cada (download de ~280 MB e ~470 MB + varredura de ~17 mil imagens + 8 épocas). Se a sessão cair, os resultados de cada dataset já concluído ficam salvos no Drive.

> ⚠️ **Este é um piloto de comparação, não o trabalho final.** A mesma configuração simples é aplicada aos três datasets, sem otimizar nada. **O conjunto de teste é separado aqui e nunca é carregado**: todas as decisões usam apenas a validação.

## 0. Integrantes e declaração de uso de IA Generativa

*(A lauda exige esta seção no início do notebook final. Se este piloto for anexado ao trabalho, preencha-a aqui também.)*

**Integrantes:** _(nome 1)_, _(nome 2)_, _(nome 3)_, _(nome 4)_

**Declaração de uso de IA Generativa**

1. **Ferramenta utilizada:** Claude Code (Anthropic), modelo Claude Opus 5.5.
2. **Finalidade do uso:** gerar o código e o texto explicativo deste notebook-piloto de comparação de datasets e inspecionar previamente a estrutura dos arquivos públicos de cada dataset.
3. **Extensão aproximada da contribuição:** _(preencher pela equipe — ex.: "código do piloto gerado por IA e revisado pela equipe; interpretação dos resultados e decisão final feitas pela equipe")_.
4. **Medidas adotadas para validação:** _(preencher — ex.: execução completa no Colab, conferência das contagens com os artigos originais, inspeção visual das duplicatas e dos mapas Grad-CAM, revisão de cada célula pela equipe)_.

## Leitura da lauda → o que este piloto verifica

| Exigência da lauda | Como o piloto trata |
|---|---|
| ≥ 1.000 imagens e ≥ 100 por classe | Contagem automática por classe e razão de desbalanceamento (Etapa 1) |
| Repositório confiável; **vedado** dataset "amplamente documentado"; proposta validada em aula | Checagem do DOI e da **integridade do arquivo baixado**; nível de documentação estimado no critério (a). **A palavra final é da professora**, então leve o resultado deste piloto para validar a proposta |
| Contexto, o que se classifica e finalidade | Critério (g), na comparação final |
| Holdout estratificado treino/validação/teste, **teste isolado desde o início** | Divisão 70/15/15 **estratificada e por grupos** (folha, rajada de fotos, duplicatas), salva em CSV para reutilizar no trabalho final. O teste **nunca** é carregado (`assert` no código) |
| Redimensionamento, canais e pré-processamento dos pesos | `preprocess_input` oficial da rede, 224×224, RGB de 3 canais |
| Métricas coerentes com desbalanceamento; quais erros são mais graves | F1 macro e *balanced accuracy* com intervalo de confiança por bootstrap; matriz de confusão normalizada; critério (g) |
| Base congelada como condição de referência | 5 épocas com a base congelada |
| Fine-tuning: parte da base precisa ser ajustada; A e B partem de cópias do mesmo modelo salvo | O piloto descongela o último bloco por 3 épocas (só mede a margem de ganho). No trabalho final: salvar os pesos da etapa 3 e criar A e B a partir de cópias deles |
| Análise visual de erros corrigidos, persistentes e novos | Contagem dessas transições na validação (base congelada → FT) |
| Grad-CAM comparando as condições, **identificando camada e classe** | Mesmas 4 imagens nas duas condições do piloto, com camada e classe investigada no título |
| Imagem externa da internet | Critério (i) |
| Arquitetura: até 2 equipes por família, com versões diferentes | `BACKBONE` configurável. **Rode o piloto com a arquitetura que sua equipe inscreveu** |
| Todas as células executam em ordem, dependências informadas | Seeds fixas, versões registradas, caminhos manuais de fallback |

## O que já foi verificado ao preparar este notebook (13/11/2026)

Antes de escrever o código, inspecionei os arquivos públicos de cada dataset. O notebook **re-verifica tudo** isso ao rodar, mas convém já saber:

**BRACOL**
- O único arquivo do DOI no Mendeley (`BRACOL_coffee_leaf_ images_datasets.zip`, 164,5 MB) **bate com o sha256 publicado, mas está truncado**: não tem diretório central de zip, contém só ~1.400 das 1.747 imagens de folhas inteiras e **não contém a pasta `symptom`**. A célula de auditoria abaixo mostra isso.
- A versão *symptom* está no GitHub dos autores ([esgario/lara2018](https://github.com/esgario/lara2018), `classification/dataset/symptom`), com divisão própria train/val/test: **2.722 imagens em 5 classes**.
- Segundo o README dos autores, essa versão **mistura imagens deles com imagens do repositório Digipathos (Embrapa)**. Pelos nomes, 592 imagens (22%) vêm do Digipathos (`ferr*`, `cerc*`, `bmin*`, `phom*`), e **nenhuma é da classe saudável**. Isso abre espaço para um atalho de "fonte" (estilo da foto → classe).
- As demais imagens são **recortes de folhas** com nome `{id_folha}_{x}_{y}_{w}_{h}.jpg`. Há 1.344 folhas distintas, com até 8 recortes por folha, e **286 folhas aparecem em mais de um split oficial** → vazamento na divisão dos autores. Por isso o piloto divide **por folha**.

**PBC**
- O zip do Mendeley (281 MB) está íntegro (17.122 entradas). Ele inclui lixo de macOS (`__MACOSX`, `.DS_Store`), que o carregador filtra.
- A classe `ig` (granulócitos imaturos) agrega subtipos pelo prefixo do arquivo (`MY`, `MMY`, `PMY`, `IG`), e `neutrophil` agrega `BNE` (bastonete) e `SNE` (segmentado). Isso ajuda a interpretar as confusões.
- Não há identificador de paciente ou lâmina, então não dá para dividir por paciente (limitação a declarar).

**DeepWeeds**
- `labels.csv` (GitHub dos autores) tem 17.509 linhas. Os nomes dos arquivos são carimbos de data e hora (`AAAAMMDD-HHMMSS-k.jpg`): fotos com o mesmo carimbo são da **mesma rajada** e precisam ficar no mesmo split.
- As imagens ficam no Google Drive (468 MB), e o download automático pode falhar por cota. Nesse caso o notebook tenta o `tensorflow_datasets` e, se também falhar, explica como baixar manualmente.

## Configuração do ambiente

**Decisões:**
- Uso **TensorFlow/Keras**, que já vem no Colab, com MobileNetV2 por padrão (barata na T4 e com pesos ImageNet oficiais). Se a equipe inscreveu ResNet50, troque `BACKBONE`.
- **Seeds fixas** (Python, NumPy, TensorFlow). *Não* ativei `enable_op_determinism`, porque algumas operações de convolução na GPU não têm versão determinística e o notebook quebraria no fine-tuning. Por isso pode haver variação residual pequena entre execuções (tipicamente décimos de ponto percentual), e é exatamente para isso que existem os intervalos de confiança.
- Versões de todas as bibliotecas ficam registradas na tela e num JSON.

In [ ]:
# Instala apenas o que não vem no Colab (imagehash). gdown e tensorflow_datasets já vêm instalados.
!pip -q install imagehash gdown

In [ ]:
import os, sys, io, re, gc, json, time, random, shutil, hashlib, zipfile, struct, subprocess, platform, warnings
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import PIL
from PIL import Image
import imagehash
import requests
import sklearn
from sklearn.model_selection import StratifiedGroupKFold, train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import tensorflow as tf
import keras
from keras import layers
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

try:
    import tensorflow_datasets as tfds
    TFDS_V = tfds.__version__
except Exception:
    TFDS_V = "não instalado"
try:
    import gdown
    GDOWN_V = gdown.__version__
except Exception:
    GDOWN_V = "não instalado"

VERSIONS = {
    "python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__, "Pillow": PIL.__version__, "imagehash": imagehash.__version__,
    "tensorflow_datasets": TFDS_V, "gdown": GDOWN_V,
}
try:
    gpu_name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                              capture_output=True, text=True).stdout.strip()
except Exception:
    gpu_name = ""
VERSIONS["GPU"] = gpu_name or ("detectada" if tf.config.list_physical_devices("GPU")
                               else "NENHUMA — ative a T4 em Ambiente de execução > Alterar tipo")
display(pd.Series(VERSIONS, name="versão").to_frame())

### Configuração única do piloto (idêntica para os três datasets)

A comparação só é justa se **tudo for igual** entre os datasets: arquitetura, entrada, pré-processamento, otimizador, *learning rate*, *batch size* e número de épocas. Os valores abaixo são os que você pediu, e **nada foi otimizado**.

In [ ]:
# ---------------- Configuração única do piloto ----------------
BACKBONE = "mobilenet_v2"   # "mobilenet_v2" ou "resnet50" — use a família/versão inscrita pela sua equipe
IMG_SIZE = 224
BATCH = 32
EPOCHS_FROZEN = 5           # base congelada
LR_FROZEN = 1e-3
EPOCHS_FT = 3               # descongelando o último bloco
LR_FT = 1e-5
SPLIT_FRACS = (0.70, 0.15, 0.15)  # implementado como 14/3/3 de 20 folds (StratifiedGroupKFold)
HASH_THR = 6                # distância de Hamming máx. do pHash (64 bits) para CANDIDATO a duplicata
NCC_THR = 0.92              # correlação mínima entre miniaturas 48x48 para CONFIRMAR a duplicata
N_BOOT = 500                # reamostragens bootstrap para os intervalos de confiança
DATASETS_TO_RUN = ["BRACOL", "PBC", "DeepWeeds"]
AUDIT_BRACOL_MENDELEY = True  # baixa o zip do DOI (164 MB) só para documentar que está truncado

# Caminhos manuais: preencha só se o download automático falhar (instruções na Etapa 1)
BRACOL_MANUAL_DIR = None     # pasta "symptom" (com train/val/test/<classe>/ ou <classe>/)
PBC_MANUAL_DIR = None        # pasta "PBC_dataset_normal_DIB" (com <classe>/)
DEEPWEEDS_MANUAL_DIR = None  # pasta com os 17.509 .jpg do images.zip

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/piloto_datasets_cnn")
else:
    OUT_DIR = Path("piloto_out").resolve()
DATA_DIR = Path("/content/data") if IN_COLAB else Path("data").resolve()   # disco local = leitura rápida
for d_ in [OUT_DIR, OUT_DIR / "figs", OUT_DIR / "splits", DATA_DIR]:
    d_.mkdir(parents=True, exist_ok=True)

CONFIG = dict(SEED=SEED, BACKBONE=BACKBONE, IMG_SIZE=IMG_SIZE, BATCH=BATCH, EPOCHS_FROZEN=EPOCHS_FROZEN,
              LR_FROZEN=LR_FROZEN, EPOCHS_FT=EPOCHS_FT, LR_FT=LR_FT, SPLIT_FRACS=SPLIT_FRACS,
              HASH_THR=HASH_THR, NCC_THR=NCC_THR, N_BOOT=N_BOOT)
with open(OUT_DIR / "config_versions.json", "w") as f:
    json.dump({"versions": VERSIONS, "config": CONFIG}, f, indent=2)
print("Saídas em:", OUT_DIR)
print("Dados em:  ", DATA_DIR)

---
# Etapa 1 — Carregamento e conformidade

**Decisões:**
- Para o Mendeley uso a **API pública** (`data.mendeley.com/public-api/...`), que devolve a URL de download e o **sha256** de cada arquivo. Assim conferimos a integridade e não só se o download "terminou".
- BRACOL *symptom* vem do GitHub dos autores via *sparse checkout*: baixa só a pasta `symptom` (~41 MB), não o repositório inteiro.
- DeepWeeds: primeiro as imagens originais (Google Drive, via `gdown`) + `labels.csv` do GitHub, porque **preservam os nomes de arquivo** e permitem detectar rajadas de fotos. Se isso falhar, uso o `tensorflow_datasets` (que perde os nomes) e, por último, as instruções manuais.
- Cada dataset é carregado num `try/except`: se um falhar, o notebook explica o download manual e **continua com os outros**.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff"}

def list_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*")
                  if p.is_file() and p.suffix.lower() in IMG_EXTS
                  and "__MACOSX" not in p.parts and not p.name.startswith("."))

def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def download_file(url, dest, expected_sha256=None):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and (expected_sha256 is None or sha256_of(dest) == expected_sha256):
        print(f"  já existe: {dest.name}")
        return dest
    print(f"  baixando {dest.name} ...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for part in r.iter_content(1 << 20):
                f.write(part)
    if expected_sha256:
        got = sha256_of(dest)
        print(f"  sha256 {'confere' if got == expected_sha256 else 'NÃO CONFERE'} ({got[:16]}...)")
    return dest

def mendeley_files(dataset_id, version=1):
    url = f"https://data.mendeley.com/public-api/datasets/{dataset_id}/files?folder_id=root&version={version}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

def zip_scan_local_headers(path):
    # Lista entradas lendo os cabeçalhos locais — funciona mesmo em zip truncado (sem diretório central).
    data = Path(path).read_bytes()
    names, i = [], 0
    while True:
        j = data.find(b"PK\x03\x04", i)
        if j < 0:
            break
        fnlen = struct.unpack("<H", data[j + 26:j + 28])[0]
        names.append(data[j + 30:j + 30 + fnlen].decode("utf-8", "replace"))
        i = j + 30 + fnlen
    has_central_dir = data.rfind(b"PK\x05\x06") >= 0
    return names, has_central_dir

def extract_zip(path, dest):
    with zipfile.ZipFile(path) as z:
        members = [m for m in z.namelist() if not m.startswith("__MACOSX")]
        z.extractall(dest, members=members)

### Auditoria do arquivo do DOI do BRACOL (evidência para a defesa)

Por que usar o GitHub e não o DOI? Esta célula baixa o arquivo do DOI, confere o sha256 e mostra o que há dentro, para que a escolha da fonte fique **documentada**, e não apenas afirmada.

In [ ]:
BRACOL_AUDIT = {}
if AUDIT_BRACOL_MENDELEY and "BRACOL" in DATASETS_TO_RUN:
    try:
        files = mendeley_files("yy2k5y8mxg", 1)
        for f in files:
            cd = f["content_details"]
            print(f"Arquivo no DOI: {f['filename']}  ({cd['size']/1e6:.1f} MB)")
            p = download_file(cd["download_url"], DATA_DIR / "bracol_mendeley" / "bracol_doi.zip", cd["sha256_hash"])
            ok_zip = zipfile.is_zipfile(p)
            names, has_cd = zip_scan_local_headers(p)
            imgs = [n for n in names if n.lower().endswith((".jpg", ".png"))]
            folders = Counter("/".join(n.split("/")[:-1]) for n in imgs)
            BRACOL_AUDIT = {"arquivo": f["filename"], "sha256 confere": sha256_of(p) == cd["sha256_hash"],
                            "zipfile consegue abrir": ok_zip, "tem diretório central": has_cd,
                            "imagens encontradas": len(imgs), "pastas": dict(folders),
                            "contém pasta 'symptom'": any("symptom" in n.lower() for n in names)}
            display(pd.Series(BRACOL_AUDIT, name="auditoria do DOI").to_frame())
            p.unlink()   # libera espaço — não usaremos este arquivo
    except Exception as e:
        print("Auditoria não pôde ser feita:", repr(e))

### Carregadores dos três datasets (com fallback manual)

In [ ]:
# ------------------------------ BRACOL (symptom) ------------------------------
BRACOL_LABELS = {"1_health": "saudavel", "2_miner": "bicho_mineiro", "3_rust": "ferrugem",
                 "4_phoma": "mancha_phoma", "5_cercospora": "cercosporiose"}
SPLIT_NAMES = {"train", "val", "test"}

def load_bracol():
    root = DATA_DIR / "bracol"
    repo = root / "lara2018"
    sym = repo / "classification" / "dataset" / "symptom"
    if BRACOL_MANUAL_DIR:
        sym = Path(BRACOL_MANUAL_DIR)
    elif len(list_images(sym)) < 2000:
        if repo.exists():
            shutil.rmtree(repo)
        root.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                        "https://github.com/esgario/lara2018.git", str(repo)], check=True)
        subprocess.run(["git", "-C", str(repo), "sparse-checkout", "set",
                        "classification/dataset/symptom"], check=True)
    pat = re.compile(r"^(\d+)_(\d+)_(\d+)_(\d+)_(\d+)$")
    rows = []
    for p in list_images(sym):
        folder = p.parent.name
        orig = p.parent.parent.name if p.parent.parent.name in SPLIT_NAMES else "—"
        m = pat.match(p.stem)
        rows.append({"path": str(p), "label": BRACOL_LABELS.get(folder, folder), "orig_split": orig,
                     "source": "BRACOL (autores)" if m else "Digipathos (Embrapa)",
                     "group": f"folha_{m.group(1)}" if m else f"img_{p.stem}"})
    return pd.DataFrame(rows)

# ------------------------------ PBC ------------------------------
def load_pbc():
    root = DATA_DIR / "pbc"
    if PBC_MANUAL_DIR:
        base = Path(PBC_MANUAL_DIR)
    else:
        found = [p for p in root.rglob("PBC_dataset_normal_DIB") if p.is_dir()] if root.exists() else []
        if not found or len(list_images(found[0])) < 15000:
            for f in mendeley_files("snkd93bnjr", 1):
                cd = f["content_details"]
                z = download_file(cd["download_url"], root / f["filename"], cd["sha256_hash"])
                print("  extraindo...")
                extract_zip(z, root)
                z.unlink()
            found = [p for p in root.rglob("PBC_dataset_normal_DIB") if p.is_dir()]
        base = found[0]
    rows = []
    for p in list_images(base):
        m = re.match(r"^([A-Za-z]+)_", p.name)
        rows.append({"path": str(p), "label": p.parent.name,
                     "subtype": m.group(1).upper() if m else "?",
                     "group": f"img_{p.stem}"})   # sem ID de paciente: cada imagem é seu próprio grupo
    return pd.DataFrame(rows)

# ------------------------------ DeepWeeds ------------------------------
DEEPWEEDS_DRIVE_ID = "1xnK3B6K6KekDI55vwJ0vnc2IGoDga9cj"   # images.zip, link do README dos autores
DEEPWEEDS_LABELS_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels/labels.csv"

def _deepweeds_rows(img_dir, labels_csv):
    lab = pd.read_csv(labels_csv)
    files = {p.name: p for p in list_images(img_dir)}
    lab = lab[lab.Filename.isin(files)].reset_index(drop=True)
    ts = lab.Filename.str.rsplit("-", n=1).str[0]         # AAAAMMDD-HHMMSS = rajada
    return pd.DataFrame({"path": [str(files[f]) for f in lab.Filename], "label": lab.Species.values,
                         "capture_ts": ts.values, "session": lab.Filename.str[:8].values,
                         "group": ("rajada_" + ts).values})

def load_deepweeds_tfds(labels_csv):
    import tensorflow_datasets as tfds
    lab = pd.read_csv(labels_csv)
    id2name = dict(zip(lab.Label, lab.Species))
    ds = tfds.load("deep_weeds", split="train", as_supervised=True, try_gcs=True,
                   data_dir=str(DATA_DIR / "tfds"))
    out = DATA_DIR / "deepweeds_tfds"
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, (img, y) in enumerate(tfds.as_numpy(ds)):
        p = out / f"{i:05d}.jpg"
        if not p.exists():
            Image.fromarray(img).save(p, quality=95)
        rows.append({"path": str(p), "label": id2name.get(int(y), str(int(y))), "group": f"img_{i}"})
    print("  ATENÇÃO: via TFDS não há nomes de arquivo originais → sem análise de rajadas/sessões.")
    return pd.DataFrame(rows)

def load_deepweeds():
    root = DATA_DIR / "deepweeds"
    root.mkdir(parents=True, exist_ok=True)
    labels_csv = download_file(DEEPWEEDS_LABELS_URL, root / "labels.csv")
    img_dir = Path(DEEPWEEDS_MANUAL_DIR) if DEEPWEEDS_MANUAL_DIR else root / "images"
    if len(list_images(img_dir)) < 17000 and not DEEPWEEDS_MANUAL_DIR:
        try:
            import gdown
            z = root / "images.zip"
            if not zipfile.is_zipfile(z):
                gdown.download(f"https://drive.google.com/uc?id={DEEPWEEDS_DRIVE_ID}", str(z), quiet=False)
            if not zipfile.is_zipfile(z):
                raise RuntimeError("o Google Drive não devolveu um zip válido (cota excedida?)")
            print("  extraindo...")
            extract_zip(z, img_dir)
            z.unlink()
        except Exception as e:
            print("  gdown falhou:", repr(e)[:200])
            print("  tentando tensorflow_datasets ('deep_weeds')...")
            return load_deepweeds_tfds(labels_csv)
    df = _deepweeds_rows(img_dir, labels_csv)
    if len(df) == 0:
        raise RuntimeError("nenhuma imagem casou com labels.csv")
    return df

LOADERS = {"BRACOL": load_bracol, "PBC": load_pbc, "DeepWeeds": load_deepweeds}
MANUAL_HELP = {
    "BRACOL": ("Download manual do BRACOL symptom: abra https://github.com/esgario/lara2018 → Code → Download ZIP "
               "(ou o link 'Coffee datasets' do README). Extraia e envie para o Drive a pasta "
               "classification/dataset/symptom. Depois defina BRACOL_MANUAL_DIR = '/content/drive/MyDrive/.../symptom' "
               "e rode de novo a partir da célula de configuração."),
    "PBC": ("Download manual do PBC: abra https://data.mendeley.com/datasets/snkd93bnjr/1 → Download All. "
            "Extraia PBC_dataset_normal_DIB.zip, envie a pasta PBC_dataset_normal_DIB para o Drive e defina "
            "PBC_MANUAL_DIR com esse caminho."),
    "DeepWeeds": ("Download manual do DeepWeeds: baixe images.zip pelo link do README de "
                  "https://github.com/AlexOlsen/DeepWeeds (Google Drive, 468 MB) ou por um espelho no Kaggle "
                  "(busque 'DeepWeeds'). Extraia os .jpg numa pasta do Drive e defina DEEPWEEDS_MANUAL_DIR. "
                  "O labels.csv é baixado automaticamente do GitHub."),
}

In [ ]:
DFS, LOAD_STATUS = {}, {}
for name in DATASETS_TO_RUN:
    print(f"\n=== {name} ===")
    t0 = time.time()
    try:
        df_ = LOADERS[name]()
        if len(df_) == 0:
            raise RuntimeError("0 imagens encontradas")
        DFS[name] = df_.reset_index(drop=True)
        LOAD_STATUS[name] = f"ok — {len(df_)} imagens ({time.time() - t0:.0f}s)"
    except Exception as e:
        LOAD_STATUS[name] = f"FALHOU: {e!r}"[:300]
        print(LOAD_STATUS[name])
        print(MANUAL_HELP[name])
    print(LOAD_STATUS[name])
display(pd.Series(LOAD_STATUS, name="status do carregamento").to_frame())

### Varredura única de todas as imagens

Cada imagem é lida **uma vez** e dessa leitura saem: formato, modo de cor, canais, resolução, md5 (duplicatas exatas), **pHash** da imagem e das versões espelhadas/rotacionadas (quase-duplicatas e *augmentation* embutida), uma miniatura 48×48 (para confirmar duplicatas e calcular a imagem média por classe) e estatísticas de cor da **moldura** e do **centro** (para testar atalhos de fundo).

A decodificação usa `draft()` do Pillow, que decodifica o JPEG já reduzido (muito mais rápido). O resultado fica em cache no disco local.

In [ ]:
THUMB = 48
VARIANTS = ["flip_h", "flip_v", "rot90", "rot180", "rot270"]
_T = Image.Transpose
PIL_OPS = [_T.FLIP_LEFT_RIGHT, _T.FLIP_TOP_BOTTOM, _T.ROTATE_90, _T.ROTATE_180, _T.ROTATE_270]
NP_OPS = [np.fliplr, np.flipud, lambda a: np.rot90(a, 1), lambda a: np.rot90(a, 2), lambda a: np.rot90(a, 3)]

def _phash_bits(img):
    return np.asarray(imagehash.phash(img).hash, dtype=np.uint8).ravel()

def scan_one(path):
    rec = {"path": path}
    try:
        with open(path, "rb") as fh:
            raw = fh.read()
        rec["md5"] = hashlib.md5(raw).hexdigest()
        rec["file_kb"] = len(raw) / 1024
        with Image.open(io.BytesIO(raw)) as im:
            rec["format"], rec["mode"] = im.format, im.mode
            rec["width"], rec["height"] = im.size
            rec["channels"] = len(im.getbands())
            try:
                im.draft("RGB", (256, 256))
            except Exception:
                pass
            rgb = im.convert("RGB")
        small = rgb.resize((128, 128), Image.Resampling.BILINEAR)
        arr = np.asarray(small, dtype=np.float32)
        b = 12
        border = np.concatenate([arr[:b].reshape(-1, 3), arr[-b:].reshape(-1, 3),
                                 arr[b:-b, :b].reshape(-1, 3), arr[b:-b, -b:].reshape(-1, 3)])
        center = arr[32:96, 32:96].reshape(-1, 3)
        for c, v in zip("rgb", border.mean(0)): rec[f"b_mean_{c}"] = float(v)
        for c, v in zip("rgb", border.std(0)):  rec[f"b_std_{c}"] = float(v)
        for c, v in zip("rgb", center.mean(0)): rec[f"c_mean_{c}"] = float(v)
        rec["brightness"] = float(arr.mean())
        rec["contrast"] = float(arr.std())
        thumb = np.asarray(small.resize((THUMB, THUMB), Image.Resampling.BILINEAR), dtype=np.uint8)
        bits = _phash_bits(small)
        vbits = np.stack([_phash_bits(small.transpose(op)) for op in PIL_OPS])
        rec["ok"] = True
        return rec, thumb, bits, vbits
    except Exception as e:
        rec["ok"] = False
        rec["error"] = repr(e)
        return rec, None, None, None

def scan_dataset(name, workers=8):
    df = DFS[name]
    cache = DATA_DIR / f"scan_{name}.pkl"
    if cache.exists():
        S = pd.read_pickle(cache)
        if len(S["df"]) + S["n_failed"] == len(df) and set(S["df"].path) <= set(df.path):
            print(f"{name}: varredura carregada do cache ({len(S['df'])} imagens)")
            return S
    t0 = time.time()
    recs, thumbs, bits, vbits = [], [], [], []
    with ThreadPoolExecutor(workers) as ex:
        for k, (rec, th, bt, vb) in enumerate(ex.map(scan_one, df.path.tolist())):
            recs.append(rec)
            if rec["ok"]:
                thumbs.append(th); bits.append(bt); vbits.append(vb)
            if (k + 1) % 3000 == 0:
                print(f"  {name}: {k + 1}/{len(df)} ({time.time() - t0:.0f}s)")
    meta = pd.DataFrame(recs)
    failed = meta[~meta.ok]
    if len(failed):
        print(f"  {len(failed)} arquivos ilegíveis descartados, ex.: {failed.path.head(3).tolist()}")
    good = meta[meta.ok].drop(columns=[c for c in ["ok", "error"] if c in meta.columns])
    merged = df.merge(good, on="path", how="inner").reset_index(drop=True)
    S = {"df": merged, "thumbs": np.stack(thumbs), "bits": np.stack(bits), "vbits": np.stack(vbits),
         "n_failed": int(len(failed))}
    pd.to_pickle(S, cache)
    print(f"{name}: {len(merged)} imagens varridas em {time.time() - t0:.0f}s")
    return S

SCANS = {name: scan_dataset(name) for name in DFS}

### Conformidade com a lauda e características técnicas

In [ ]:
def compliance_row(name):
    d = SCANS[name]["df"]
    cnt = d.label.value_counts()
    return {
        "imagens": len(d), "classes": len(cnt),
        "maior classe": f"{cnt.idxmax()} ({cnt.max()})", "menor classe": f"{cnt.idxmin()} ({cnt.min()})",
        "razão maior/menor": round(cnt.max() / cnt.min(), 2),
        "cumpre ≥1.000 imagens": bool(len(d) >= 1000), "cumpre ≥100 por classe": bool(cnt.min() >= 100),
        "formatos": dict(Counter(d.format)), "modos de cor": dict(Counter(d["mode"])),
        "nº de canais": sorted(d.channels.unique().tolist()),
        "largura (mín–máx)": f"{d.width.min()}–{d.width.max()}",
        "altura (mín–máx)": f"{d.height.min()}–{d.height.max()}",
        "resoluções distintas": int(d.groupby(["width", "height"]).ngroups),
        "arquivos ilegíveis": SCANS[name]["n_failed"],
    }

COMPLIANCE = {name: compliance_row(name) for name in SCANS}
display(pd.DataFrame(COMPLIANCE))

for name in SCANS:
    d = SCANS[name]["df"]
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
    cnt = d.label.value_counts()
    ax[0].barh(cnt.index[::-1], cnt.values[::-1], color="#4C72B0")
    ax[0].axvline(100, color="red", ls="--", lw=1, label="mínimo da lauda (100)")
    ax[0].set_title(f"{name} — imagens por classe"); ax[0].legend(fontsize=8)
    top = d.groupby(["width", "height"]).size().sort_values(ascending=False).head(8)
    ax[1].barh([f"{w}x{h}" for w, h in top.index][::-1], top.values[::-1], color="#55A868")
    ax[1].set_title(f"{name} — resoluções mais frequentes")
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / f"{name}_contagens.png", dpi=110); plt.show()

**📝 Interpretação (preencher após rodar):** algum dataset fica abaixo de 100 imagens na menor classe? Qual tem o maior desbalanceamento? Se a razão maior/menor for alta (DeepWeeds, por causa da classe *Negative*), acurácia simples engana, e por isso o piloto usa F1 macro e *balanced accuracy*. Resoluções muito diferentes **por classe** são um sinal de alerta (ver Etapa 2).

---
# Etapa 2 — Qualidade dos dados, vazamento e vieses

**Decisões e por quê:**
1. **Grade com 5 amostras por classe:** inspeção visual de fundo, iluminação, bordas e marcas d'água.
2. **Duplicatas exatas** (md5) e **quase-duplicatas** em dois estágios: (i) pHash com distância de Hamming ≤ `HASH_THR` gera os *candidatos*; (ii) a correlação normalizada entre miniaturas 48×48 ≥ `NCC_THR` *confirma*. O segundo estágio existe porque o pHash sozinho gera falsos positivos em imagens muito parecidas por natureza (ex.: células centralizadas sobre fundo rosa). Exemplos são exibidos para conferência visual.
3. ***Augmentation* embutida:** comparo o pHash de cada imagem espelhada/rotacionada (5 variantes) com o das outras imagens. Se B ≈ flip/rotação de A (e B não era já uma duplicata comum de A), é forte indício de cópia aumentada no dataset original. Também procuro nomes de arquivo com `aug`, `flip`, `rot` e `copy`.
4. **Divisão 70/15/15 por grupos:** imagens do mesmo grupo (mesma folha no BRACOL, mesma rajada no DeepWeeds, ou ligadas por duplicata) vão **sempre para o mesmo split**. Comparo com a divisão estratificada "ingênua" para **quantificar** o vazamento que ela causaria. Uso `StratifiedGroupKFold` com 20 folds: 3 para teste, 3 para validação e 14 para treino (≈ 15/15/70).
5. **Vieses e atalhos:** além da inspeção visual, um **teste quantitativo**: uma regressão logística treinada **só com a cor da moldura da imagem** (12 px de borda), só com resolução/tamanho do arquivo ou só com cor global. Se esses atributos triviais previrem a classe muito acima do acaso, existe um **atalho** que a CNN também pode aprender. O teste usa apenas o treino, com validação cruzada por grupos.

In [ ]:
def show_grid(name, n=5):
    d = SCANS[name]["df"]
    classes = sorted(d.label.unique())
    rng = np.random.default_rng(SEED)
    fig, axes = plt.subplots(len(classes), n, figsize=(2.1 * n + 1.6, 2.1 * len(classes)))
    for r, c in enumerate(classes):
        sub = d[d.label == c]
        pick = rng.choice(len(sub), size=min(n, len(sub)), replace=False)
        for k in range(n):
            ax = axes[r, k]
            ax.set_xticks([]); ax.set_yticks([])
            if k < len(pick):
                row = sub.iloc[pick[k]]
                ax.imshow(Image.open(row.path).convert("RGB"))
                ax.set_title(f"{row.width}x{row.height}", fontsize=7)
        axes[r, 0].set_ylabel(c, fontsize=9, rotation=0, ha="right", va="center")
    fig.suptitle(f"{name} — {n} amostras aleatórias por classe (seed {SEED})", fontsize=11)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / f"{name}_grade.png", dpi=110); plt.show()

def find_pairs(A_bits, B_bits, thr, upper=False, exclude_diag=True, chunk=2048, max_pairs=3_000_000):
    # Pares (i, j) com distância de Hamming <= thr entre linhas de A e B (bits 0/1), por multiplicação de matrizes.
    A = A_bits.astype(np.float32); B = B_bits.astype(np.float32)
    sa, sb = A.sum(1), B.sum(1)
    res, total = [], 0
    for s in range(0, len(A), chunk):
        dist = sa[s:s + chunk, None] + sb[None, :] - 2.0 * (A[s:s + chunk] @ B.T)
        ii, jj = np.nonzero(dist <= thr + 0.5)
        ii = ii + s
        keep = (jj > ii) if upper else ((ii != jj) if exclude_diag else np.ones(len(ii), bool))
        res.append(np.stack([ii[keep], jj[keep]], 1))
        total += int(keep.sum())
        if total > max_pairs:
            print(f"  aviso: mais de {max_pairs} candidatos — interrompido; reduza HASH_THR")
            break
    return np.concatenate(res).astype(int) if res else np.zeros((0, 2), int)

def ncc_pairs(G, pairs, op=None, chunk=50_000):
    # Correlação normalizada entre miniaturas em tons de cinza (op transforma a 1ª imagem do par).
    out = []
    for s in range(0, len(pairs), chunk):
        p = pairs[s:s + chunk]
        a = G[p[:, 0]]
        if op is not None:
            a = np.stack([op(x) for x in a])
        b = G[p[:, 1]]
        a = a.reshape(len(a), -1); b = b.reshape(len(b), -1)
        a = a - a.mean(1, keepdims=True); b = b - b.mean(1, keepdims=True)
        out.append((a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-6))
    return np.concatenate(out) if out else np.zeros(0)

def build_groups(n, pairs, meta_groups):
    parent = np.arange(n)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
    for i, j in pairs:
        union(int(i), int(j))
    for idx in pd.Series(np.arange(n)).groupby(np.asarray(meta_groups)).groups.values():
        idx = list(idx)
        for k in idx[1:]:
            union(idx[0], k)
    return np.array([f"g{find(i)}" for i in range(n)])

def make_splits(y, groups, seed=SEED):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=seed)
    naive = np.empty(len(y), dtype=object)
    naive[tr], naive[va], naive[te] = "train", "val", "test"
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    fold = np.full(len(y), -1)
    for k, (_, f) in enumerate(sgkf.split(idx, y, groups)):
        fold[f] = k
    grouped = np.where(fold < 3, "test", np.where(fold < 6, "val", "train"))
    return naive.astype(str), grouped

def cross_stats(tag, split, pairs, meta_groups):
    s = np.asarray(split).astype(str)
    out = {"divisão": tag}
    if len(pairs):
        a, b = s[pairs[:, 0]], s[pairs[:, 1]]
        out["pares duplicados treino/val ↔ teste"] = int(((a != b) & ((a == "test") | (b == "test"))).sum())
        out["pares duplicados treino ↔ val"] = int((((a == "train") & (b == "val")) | ((a == "val") & (b == "train"))).sum())
    else:
        out["pares duplicados treino/val ↔ teste"] = 0
        out["pares duplicados treino ↔ val"] = 0
    gs = pd.DataFrame({"g": meta_groups, "s": s}).groupby("g").s.agg(lambda x: frozenset(x))
    multi = gs[gs.map(len) > 1]
    out["grupos de origem em >1 split"] = int(len(multi))
    out["... que tocam o teste"] = int(sum("test" in v for v in multi))
    return out

def show_pairs(d, pairs, title, k=6):
    pairs = np.asarray(pairs)[:k]
    if len(pairs) == 0:
        print(f"{title}: nenhum par.")
        return
    fig, axes = plt.subplots(len(pairs), 2, figsize=(5.2, 2.5 * len(pairs)))
    axes = np.atleast_2d(axes)
    for r, (i, j) in enumerate(pairs):
        for c, idx in enumerate((i, j)):
            row = d.iloc[int(idx)]
            ax = axes[r, c]
            ax.imshow(Image.open(row.path).convert("RGB")); ax.axis("off")
            ax.set_title(f"{row.label}\n{Path(row.path).name[:30]}", fontsize=7)
    fig.suptitle(title, fontsize=10); plt.tight_layout(); plt.show()

QUALITY = {}

In [ ]:
def quality_report(name):
    S = SCANS[name]; d = S["df"]; n = len(d); y = d.label.values
    G = S["thumbs"].astype(np.float32).mean(-1)
    display(Markdown(f"### {name} — duplicatas, *augmentation* embutida e divisão"))

    # 1) duplicatas exatas (mesmos bytes)
    dup_md5 = d[d.duplicated("md5", keep=False)]
    exact_cross = int((dup_md5.groupby("md5").label.nunique() > 1).sum()) if len(dup_md5) else 0

    # 2) quase-duplicatas: pHash (candidatos) + NCC (confirmação)
    cand = find_pairs(S["bits"], S["bits"], HASH_THR, upper=True)
    near = cand[ncc_pairs(G, cand) >= NCC_THR] if len(cand) else cand
    in_dup = np.zeros(n, bool); in_dup[near.ravel()] = True
    near_cross = near[y[near[:, 0]] != y[near[:, 1]]] if len(near) else near

    # 3) augmentation embutida: B ~ flip/rotação de A (e que não seja já duplicata comum)
    near_set = {tuple(p) for p in np.sort(near, 1)} if len(near) else set()
    aug_list, aug_counts = [], {}
    for k, (vn, op) in enumerate(zip(VARIANTS, NP_OPS)):
        c = find_pairs(S["vbits"][:, k, :], S["bits"], HASH_THR, exclude_diag=True)
        if len(c):
            c = c[ncc_pairs(G, c, op) >= NCC_THR]
            c = np.array([p for p in c if tuple(sorted(p)) not in near_set], dtype=int).reshape(-1, 2)
        aug_counts[vn] = int(len(c)); aug_list.append(c)
    aug = (np.unique(np.sort(np.concatenate(aug_list), 1), axis=0)
           if sum(aug_counts.values()) else np.zeros((0, 2), int))
    in_aug = np.zeros(n, bool); in_aug[aug.ravel()] = True
    stems = d.path.map(lambda p: Path(p).stem.lower())
    kw = stems.str.contains(r"(?:^|[_\-])(?:aug|flip|rot\d*|copy|dup)(?:[_\-]|\d|$)", regex=True)

    summ = pd.Series({
        "imagens em duplicatas exatas (md5)": len(dup_md5),
        "grupos exatos com rótulos diferentes": exact_cross,
        f"candidatos pHash (Hamming ≤ {HASH_THR})": len(cand),
        f"pares quase-duplicados confirmados (NCC ≥ {NCC_THR})": len(near),
        "imagens envolvidas em quase-duplicatas": int(in_dup.sum()),
        "% do dataset em quase-duplicatas": round(100 * in_dup.mean(), 2),
        "pares quase-duplicados com RÓTULOS DIFERENTES": len(near_cross),
        "pares espelho/rotação (indício de augmentation)": len(aug),
        **{f"   · {k}": v for k, v in aug_counts.items()},
        "nomes de arquivo sugerindo augmentation": int(kw.sum()),
    }, name=name)
    display(summ.to_frame())
    show_pairs(d, near, f"{name}: quase-duplicatas confirmadas (confira visualmente!)")
    if len(near_cross):
        show_pairs(d, near_cross, f"{name}: quase-duplicatas com rótulos DIFERENTES (ruído de rótulo?)")
    if len(aug):
        show_pairs(d, aug, f"{name}: pares espelho/rotação (augmentation no original?)")

    # 4) grupos e divisões
    all_pairs = np.concatenate([near, aug]) if len(aug) else near
    groups = build_groups(n, all_pairs, d.group.values)
    d["split_group"] = groups
    naive, grouped = make_splits(y, groups)
    stats = []
    if "orig_split" in d and (d.orig_split != "—").all():
        stats.append(cross_stats("split oficial dos autores", d.orig_split.values, all_pairs, d.group.values))
    stats.append(cross_stats("estratificada ingênua (70/15/15)", naive, all_pairs, d.group.values))
    stats.append(cross_stats("estratificada POR GRUPOS (usada)", grouped, all_pairs, d.group.values))
    display(Markdown("**Risco de vazamento por tipo de divisão** (grupos de origem = folha no BRACOL, rajada no DeepWeeds):"))
    display(pd.DataFrame(stats).set_index("divisão"))
    assert stats[-1]["pares duplicados treino/val ↔ teste"] == 0 and stats[-1]["... que tocam o teste"] == 0

    d["split"] = grouped
    prop = pd.crosstab(d.label, d.split, normalize="columns").round(3)[["train", "val", "test"]]
    prop.loc["(n imagens)"] = d.split.value_counts()[["train", "val", "test"]].values
    display(Markdown("**Proporção das classes em cada split** (estratificação preservada?):"))
    display(prop)
    out_csv = OUT_DIR / "splits" / f"{name}_split_seed{SEED}.csv"
    d[["path", "label", "group", "split_group", "split"]].to_csv(out_csv, index=False)
    print(f"Divisão salva em {out_csv} — REUTILIZE este arquivo no trabalho final; o teste não foi lido.")

    naive_stats = stats[-2]
    QUALITY[name] = {
        "n": n, "exact_dups": len(dup_md5), "exact_cross": exact_cross,
        "near_pairs": len(near), "frac_in_dup": float(in_dup.mean()), "near_cross": len(near_cross),
        "aug_pairs": len(aug), "frac_in_aug": float(in_aug.mean()), "aug_kw": int(kw.sum()),
        "naive_pairs_cross_test": naive_stats["pares duplicados treino/val ↔ teste"],
        "naive_groups_cross_test": naive_stats["... que tocam o teste"],
        "official_groups_cross": stats[0]["grupos de origem em >1 split"] if len(stats) == 3 else None,
        "n_meta_groups_multi": int((d.group.value_counts() > 1).sum()),
        "split_sizes": d.split.value_counts().to_dict(),
    }

In [ ]:
SUBGROUP_COLS = {"BRACOL": ["source"], "PBC": ["subtype"], "DeepWeeds": []}

def bias_report(name):
    S = SCANS[name]; d = S["df"]; y = d.label.values
    train = (d.split == "train").values
    classes = sorted(d.label.unique()); K = len(classes)
    display(Markdown(f"### {name} — vieses e possíveis atalhos"))

    # (a) resolução, brilho e tamanho de arquivo por classe
    res = d.groupby("label").agg(n=("path", "size"), largura_mediana=("width", "median"),
                                 altura_mediana=("height", "median"), largura_min=("width", "min"),
                                 largura_max=("width", "max"), brilho_medio=("brightness", "mean"),
                                 kb_mediano=("file_kb", "median")).round(1)
    res["resoluções_distintas"] = d.groupby("label").apply(lambda g: g.groupby(["width", "height"]).ngroups)
    display(Markdown("**Resolução, brilho e tamanho de arquivo por classe** (diferenças sistemáticas = possível atalho):"))
    display(res)

    # (b) subgrupos (fonte das imagens, subtipo celular, sessões de captura)
    for col in SUBGROUP_COLS.get(name, []):
        if col in d:
            display(Markdown(f"**Distribuição de `{col}` por classe:**"))
            display(pd.crosstab(d[col], d.label, margins=True))
    if "session" in d:
        ses = d.groupby("label").session.agg(n_sessoes="nunique",
                                             pct_na_maior_sessao=lambda s: round(100 * s.value_counts().iloc[0] / len(s), 1))
        display(Markdown("**Sessões de captura (dia) por classe** (poucas sessões = risco de aprender o dia/local):"))
        display(ses)

    # (c) teste quantitativo de atalhos: atributos triviais preveem a classe?
    feats = {
        "só moldura (cor média/desvio de 12 px de borda)": ["b_mean_r", "b_mean_g", "b_mean_b", "b_std_r", "b_std_g", "b_std_b"],
        "só resolução e tamanho do arquivo": ["width", "height", "file_kb"],
        "cor global (centro + brilho/contraste)": ["c_mean_r", "c_mean_g", "c_mean_b", "brightness", "contrast"],
    }
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows, probe = [], {}
    for tag, cols in feats.items():
        X = d.loc[train, cols].values.astype(float)
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
        sc = cross_val_score(clf, X, y[train], groups=d.split_group.values[train], cv=cv, scoring="balanced_accuracy")
        probe[tag] = float(sc.mean())
        rows.append({"atributos triviais": tag, "balanced acc (CV, só treino)": f"{sc.mean():.3f} ± {sc.std():.3f}",
                     "acaso (1/K)": round(1 / K, 3), "× acaso": round(sc.mean() * K, 1)})
    display(Markdown("**Teste de atalho** — quanto um classificador linear acerta usando só atributos triviais:"))
    display(pd.DataFrame(rows).set_index("atributos triviais"))

    # (d) imagem média por classe (marcas d'água, bordas, enquadramento fixo aparecem aqui)
    fig, axes = plt.subplots(1, K, figsize=(2.0 * K, 2.4))
    for ax, c in zip(np.atleast_1d(axes), classes):
        ax.imshow(S["thumbs"][d.label.values == c].mean(0).astype(np.uint8))
        ax.set_title(c, fontsize=8); ax.axis("off")
    fig.suptitle(f"{name} — imagem média por classe (48x48)", fontsize=10)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / f"{name}_media_classe.png", dpi=110); plt.show()

    QUALITY[name].update({"probe_border": probe[list(feats)[0]], "probe_size": probe[list(feats)[1]],
                          "probe_color": probe[list(feats)[2]], "chance": 1 / K})

def run_if(name, fn, *a, **k):
    if name not in SCANS:
        print(f"{name} indisponível ({LOAD_STATUS.get(name, 'não carregado')}) — pulando.")
        return None
    return fn(name, *a, **k)

## 2.1 — BRACOL

Observe a tabela `source × classe`: se a classe **saudável não tiver nenhuma imagem do Digipathos**, o modelo pode aprender "estilo Digipathos ⇒ doente". Compare também o vazamento da **divisão oficial dos autores** com o da divisão por folha.

In [ ]:
run_if("BRACOL", show_grid)
run_if("BRACOL", quality_report)
run_if("BRACOL", bias_report)

## 2.2 — PBC

Observe se a moldura (hemácias e fundo) prevê a classe: parte disso é legítimo (o tamanho da célula muda o que sobra de fundo), parte pode ser lote de coloração ou aquisição. Veja também como os subtipos (`MY`, `MMY`, `PMY` em `ig`; `BNE` e `SNE` em `neutrophil`) se distribuem.

In [ ]:
run_if("PBC", show_grid)
run_if("PBC", quality_report)
run_if("PBC", bias_report)

## 2.3 — DeepWeeds

Observe a classe *Negative* (cerca de metade do dataset), o número de **sessões de captura por espécie** (espécies coletadas em poucos dias e locais facilitam um atalho de "local") e quantas rajadas cruzariam treino/teste numa divisão ingênua.

In [ ]:
run_if("DeepWeeds", show_grid)
run_if("DeepWeeds", quality_report)
run_if("DeepWeeds", bias_report)

**📝 Interpretação (preencher após rodar), para cada dataset:**
- Duplicatas: quantas, e **quantas cruzariam treino/teste** numa divisão ingênua? Há duplicatas com rótulos diferentes (ruído de rótulo)?
- *Augmentation* embutida: há pares espelho/rotação? Se sim, uma divisão aleatória põe a imagem no treino e a cópia girada no teste, o que infla o resultado.
- Atalhos: a moldura sozinha acerta muito acima do acaso? A resolução difere por classe? A imagem média mostra marca d'água, borda preta ou enquadramento fixo?
- Esses achados entram no critério (c) e nos riscos para a defesa.

---
# Etapa 3 — Piloto de transfer learning (apenas validação)

**Configuração idêntica para os três datasets e por quê:**
- **Arquitetura:** `BACKBONE` com pesos ImageNet, `include_top=False`, seguida de *GlobalAveragePooling* → *Dropout* 0,2 → *Dense* softmax. É a cabeça mínima padrão, sem nada a otimizar.
- **Entrada 224×224**, RGB de 3 canais. Redimensionamento direto (bilinear com *antialias*), sem preservar a proporção, para todos. Imagens em tons de cinza ou RGBA são convertidas para 3 canais na decodificação.
- **Pré-processamento oficial** (`keras.applications.<rede>.preprocess_input`): na MobileNetV2 escala para [-1, 1]; na ResNet50, RGB→BGR com subtração da média do ImageNet. Usar outro pré-processamento descalibraria os pesos pré-treinados.
- **Sem *data augmentation*, sem *class weights*, sem *early stopping*.** Qualquer um desses mudaria a comparação; no trabalho final eles podem ser decisões justificadas.
- **Adam, lr = 1e-3, batch 32, 5 épocas** com a base congelada. Depois, **mais 3 épocas com lr = 1e-5** descongelando o **último bloco** (MobileNetV2: `block_16` + `Conv_1`; ResNet50: `conv5_block3`). As camadas de *BatchNormalization* continuam congeladas e a base roda com `training=False`. Isso evita que as estatísticas do ImageNet sejam destruídas com um *batch* pequeno, que é o cuidado padrão no fine-tuning.
- **Métricas na validação:** acurácia, **F1 macro** (trata todas as classes por igual) e ***balanced accuracy*** (média dos *recalls* e robusta ao desbalanceamento), cada uma com **IC 95% por bootstrap**. Com validações pequenas (BRACOL ≈ 400 imagens), diferenças de 1–2 pontos podem ser ruído.
- **Tempo por época:** a 1ª época inclui a decodificação e o cache das imagens, por isso também reporto a mediana das épocas seguintes.
- **O teste nunca é carregado.** A função que monta os dados recusa qualquer split que não seja `train` ou `val`.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
BACKBONES = {
    "mobilenet_v2": (keras.applications.MobileNetV2, keras.applications.mobilenet_v2.preprocess_input, "block_16_expand"),
    "resnet50": (keras.applications.ResNet50, keras.applications.resnet50.preprocess_input, "conv5_block3_1_conv"),
}
BaseNet, PREPROCESS, FT_START = BACKBONES[BACKBONE]

def load_resize(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear", antialias=True)
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)

def make_ds(frame, split, class_to_id, training):
    assert split in ("train", "val"), "o conjunto de teste não é usado no piloto"
    sub = frame[frame.split == split]
    ds = tf.data.Dataset.from_tensor_slices((sub.path.values, sub.label.map(class_to_id).values.astype("int32")))
    ds = ds.map(lambda p, y: (load_resize(p), y), num_parallel_calls=AUTOTUNE).cache()   # cache em uint8
    if training:
        ds = ds.shuffle(len(sub), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH).map(lambda x, y: (PREPROCESS(tf.cast(x, tf.float32)), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE), sub

def build_model(n_classes):
    base = BaseNet(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    head = keras.Sequential([layers.GlobalAveragePooling2D(name="gap"),
                             layers.Dropout(0.2, seed=SEED, name="dropout"),
                             layers.Dense(n_classes, activation="softmax", name="pred")], name="head")
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    out = head(base(inp, training=False))   # BN sempre em modo inferência
    return keras.Model(inp, out, name=f"piloto_{BACKBONE}"), base, head

def unfreeze_last_block(base):
    base.trainable = True
    names = [l.name for l in base.layers]
    start = names.index(FT_START)
    for i, l in enumerate(base.layers):
        l.trainable = (i >= start) and not isinstance(l, layers.BatchNormalization)

def n_trainable(model):
    return int(sum(np.prod(v.shape) for v in model.trainable_weights))

class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.perf_counter() - self.t0)

def metrics_with_ci(y, pred, K):
    labels = list(range(K))
    m = {"acc": accuracy_score(y, pred),
         "f1_macro": f1_score(y, pred, average="macro", labels=labels, zero_division=0),
         "bal_acc": balanced_accuracy_score(y, pred),
         "f1_per_class": f1_score(y, pred, average=None, labels=labels, zero_division=0).tolist()}
    rng = np.random.default_rng(SEED); n = len(y); bf, bb = [], []
    for _ in range(N_BOOT):
        i = rng.integers(0, n, n)
        bf.append(f1_score(y[i], pred[i], average="macro", labels=labels, zero_division=0))
        bb.append(balanced_accuracy_score(y[i], pred[i]))
    m["f1_macro_ci"] = np.percentile(bf, [2.5, 97.5]).tolist()
    m["bal_acc_ci"] = np.percentile(bb, [2.5, 97.5]).tolist()
    return m

def paired_delta_ci(y, p1, p2, K):
    labels = list(range(K)); rng = np.random.default_rng(SEED); n = len(y); ds_ = []
    for _ in range(N_BOOT):
        i = rng.integers(0, n, n)
        ds_.append(f1_score(y[i], p2[i], average="macro", labels=labels, zero_division=0)
                   - f1_score(y[i], p1[i], average="macro", labels=labels, zero_division=0))
    return np.percentile(ds_, [2.5, 97.5]).tolist()

def gradcam(base, head, x_pre, class_idx):
    # Grad-CAM na saída da última camada convolucional da base (mapa 7x7), sobre o LOGIT da classe.
    W, b = head.get_layer("pred").get_weights()
    W, b = tf.constant(W), tf.constant(b)
    x = tf.convert_to_tensor(np.asarray(x_pre)[None], dtype=tf.float32)
    with tf.GradientTape() as tape:
        fmap = base(x, training=False)
        tape.watch(fmap)
        logits = tf.matmul(tf.reduce_mean(fmap, axis=(1, 2)), W) + b
        score = logits[:, class_idx]
    grads = tape.gradient(score, fmap)[0]
    weights = tf.reduce_mean(grads, axis=(0, 1))
    cam = tf.nn.relu(tf.reduce_sum(fmap[0] * weights, axis=-1)).numpy()
    return (cam / (cam.max() + 1e-8)).astype(np.float32)

def overlay(img, cam):
    cam_r = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE), method="bilinear").numpy()[..., 0]
    return np.clip(0.55 * img / 255.0 + 0.45 * plt.cm.jet(cam_r)[..., :3], 0, 1), cam_r

def center_energy(cam_r):
    h, w = cam_r.shape
    return float(cam_r[h // 4:3 * h // 4, w // 4:3 * w // 4].sum() / (cam_r.sum() + 1e-8))

def select_cam_images(y, prob):
    pred, conf = prob.argmax(1), prob.max(1)
    rng = np.random.default_rng(SEED)
    wrong = np.where(pred != y)[0]; right = np.where(pred == y)[0]
    errs, seen = [], set()
    for i in wrong[np.argsort(-conf[wrong])]:          # erros mais confiantes, de classes reais distintas
        if y[i] not in seen:
            errs.append(int(i)); seen.add(y[i])
        if len(errs) == 2: break
    for i in wrong[np.argsort(-conf[wrong])]:
        if len(errs) < 2 and int(i) not in errs: errs.append(int(i))
    cors, seen = [], set()
    for i in rng.permutation(right):                    # acertos aleatórios de classes distintas
        if y[i] not in seen:
            cors.append(int(i)); seen.add(y[i])
        if len(cors) == 2: break
    kinds = ["acerto"] * len(cors) + ["erro"] * len(errs)
    if len(errs) < 2:                                    # lauda: sem erros, analisar os casos mais incertos
        extra = [int(i) for i in right[np.argsort(conf[right])] if int(i) not in cors][:2 - len(errs)]
        errs += extra; kinds += ["acerto incerto (faltaram erros)"] * len(extra)
    return np.array(cors + errs), kinds

def plot_curves(name, h1, h2):
    H = {k: h1.history[k] + h2.history[k] for k in h1.history}
    ep = np.arange(1, len(H["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    for a, key, lab in [(ax[0], "loss", "loss"), (ax[1], "accuracy", "acurácia")]:
        a.plot(ep, H[key], "o-", label="treino"); a.plot(ep, H["val_" + key], "s-", label="validação")
        a.axvline(EPOCHS_FROZEN + 0.5, ls="--", c="gray")
        a.text(EPOCHS_FROZEN + 0.6, a.get_ylim()[1], " fine-tuning →", va="top", fontsize=8, color="gray")
        a.set_xlabel("época"); a.set_title(f"{name} — {lab}"); a.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / f"{name}_curvas.png", dpi=110); plt.show()

def plot_cms(name, y, pf, pt, classes):
    K = len(classes); cms = []
    fig, axes = plt.subplots(1, 2, figsize=(2 * max(5.5, 0.85 * K + 2.5), max(4.6, 0.75 * K + 1.6)))
    for ax, p, tag in [(axes[0], pf, "base congelada"), (axes[1], pt, "+ fine-tuning do último bloco")]:
        cm = confusion_matrix(y, p, labels=range(K), normalize="true"); cms.append(cm)
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        for i in range(K):
            for j in range(K):
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if cm[i, j] > 0.5 else "black")
        ax.set_xticks(range(K)); ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(K)); ax.set_yticklabels(classes, fontsize=8)
        ax.set_xlabel("previsto"); ax.set_ylabel("real"); ax.set_title(f"{name} — {tag} (validação, normalizada)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / f"{name}_matrizes.png", dpi=110); plt.show()
    return cms

def top_confusions(cm, classes, k=3):
    off = [(cm[i, j], classes[i], classes[j]) for i in range(len(classes)) for j in range(len(classes)) if i != j]
    return [f"{a} → {b}: {v:.1%}" for v, a, b in sorted(off, reverse=True)[:k]]

def to_py(o):
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, np.ndarray): return to_py(o.tolist())
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

RESULTS_PATH = OUT_DIR / f"pilot_results_{BACKBONE}.json"
RESULTS = json.load(open(RESULTS_PATH)) if RESULTS_PATH.exists() else {}
CAMS = {}

In [ ]:
def run_pilot(name):
    d = SCANS[name]["df"]
    if "split" not in d:
        print("rode antes a Etapa 2 deste dataset"); return
    classes = sorted(d.label.unique()); cid = {c: i for i, c in enumerate(classes)}; K = len(classes)
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    train_ds, tr = make_ds(d, "train", cid, training=True)
    val_ds, va = make_ds(d, "val", cid, training=False)
    yva = va.label.map(cid).values

    model, base, head = build_model(K)
    model.compile(optimizer=keras.optimizers.Adam(LR_FROZEN), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    p_frozen = n_trainable(model)
    print(f"{name}: treino={len(tr)}  validação={len(va)}  classes={K}  | modelo: entrada {model.input_shape} → saída {model.output_shape}, "
          f"{model.count_params():,} parâmetros, {p_frozen:,} treináveis (base congelada)")
    t1 = EpochTimer()
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FROZEN, callbacks=[t1], verbose=2)
    prob_f = model.predict(val_ds, verbose=0); pf = prob_f.argmax(1)
    m_f = metrics_with_ci(yva, pf, K)

    sel, kinds = select_cam_images(yva, prob_f)
    imgs = [load_resize(p).numpy() for p in va.path.values[sel]]
    xpre = [PREPROCESS(tf.cast(im, tf.float32)).numpy() for im in imgs]
    cams_f = [gradcam(base, head, x, int(prob_f[i].argmax())) for x, i in zip(xpre, sel)]

    unfreeze_last_block(base)
    model.compile(optimizer=keras.optimizers.Adam(LR_FT), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    p_ft = n_trainable(model)
    print(f"\nFine-tuning a partir de '{FT_START}': {p_ft:,} parâmetros treináveis (BN congeladas)")
    t2 = EpochTimer()
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FROZEN + EPOCHS_FT,
                   initial_epoch=EPOCHS_FROZEN, callbacks=[t2], verbose=2)
    prob_t = model.predict(val_ds, verbose=0); pt = prob_t.argmax(1)
    m_t = metrics_with_ci(yva, pt, K)
    cams_t = [gradcam(base, head, x, int(prob_t[i].argmax())) for x, i in zip(xpre, sel)]

    plot_curves(name, h1, h2)
    cms = plot_cms(name, yva, pf, pt, classes)

    fmt = lambda m, k: f"{m[k]:.3f} [{m[k + '_ci'][0]:.3f}–{m[k + '_ci'][1]:.3f}]" if k + "_ci" in m else f"{m[k]:.3f}"
    med = lambda t: float(np.median(t[1:])) if len(t) > 1 else float(t[0])
    tab = pd.DataFrame({
        "base congelada": [fmt(m_f, "acc"), fmt(m_f, "f1_macro"), fmt(m_f, "bal_acc"), f"{p_frozen:,}",
                           f"{t1.times[0]:.0f}s / {med(t1.times):.0f}s", EPOCHS_FROZEN],
        "+ FT último bloco": [fmt(m_t, "acc"), fmt(m_t, "f1_macro"), fmt(m_t, "bal_acc"), f"{p_ft:,}",
                              f"{t2.times[0]:.0f}s / {med(t2.times):.0f}s", EPOCHS_FT],
    }, index=["acurácia", "F1 macro [IC95%]", "balanced acc [IC95%]", "parâmetros treináveis",
              "tempo época 1 / mediana demais", "épocas"])
    display(Markdown(f"**{name} — métricas na VALIDAÇÃO**")); display(tab)
    dci = paired_delta_ci(yva, pf, pt, K)
    print(f"Ganho do FT em F1 macro: {m_t['f1_macro'] - m_f['f1_macro']:+.3f}  (IC95% pareado {dci[0]:+.3f} a {dci[1]:+.3f})")

    pc = pd.DataFrame({"F1 congelada": m_f["f1_per_class"], "F1 FT": m_t["f1_per_class"],
                       "n validação": pd.Series(yva).value_counts().reindex(range(K)).fillna(0).astype(int).values},
                      index=classes).round(3)
    pc["Δ"] = (pc["F1 FT"] - pc["F1 congelada"]).round(3)
    display(Markdown("**F1 por classe:**")); display(pc)

    cf, ct = pf == yva, pt == yva
    trans = {"acertos mantidos": int((cf & ct).sum()), "erros corrigidos pelo FT": int((~cf & ct).sum()),
             "erros persistentes": int((~cf & ~ct).sum()), "erros novos após FT": int((cf & ~ct).sum())}
    display(Markdown("**Transições de erro na validação (prévia da análise visual da lauda):**"))
    display(pd.Series(trans, name=name).to_frame())
    for col in SUBGROUP_COLS.get(name, []):
        if col in va:
            sg = pd.DataFrame({col: va[col].values, "congelada": cf, "FT": ct}).groupby(col).agg(
                n=("congelada", "size"), acc_congelada=("congelada", "mean"), acc_FT=("FT", "mean")).round(3)
            display(Markdown(f"**Acurácia por `{col}` na validação:**")); display(sg)
    print("Principais confusões (congelada):", top_confusions(cms[0], classes))
    print("Principais confusões (FT):       ", top_confusions(cms[1], classes))

    ce_f = [center_energy(overlay(im, c)[1]) for im, c in zip(imgs, cams_f)]
    ce_t = [center_energy(overlay(im, c)[1]) for im, c in zip(imgs, cams_t)]
    CAMS[name] = dict(imgs=imgs, kinds=kinds, y=yva[sel], prob_f=prob_f[sel], prob_t=prob_t[sel],
                      cams_f=cams_f, cams_t=cams_t, classes=classes, layer=base.layers[-1].name)
    RESULTS[name] = to_py({
        "backbone": BACKBONE, "classes": classes, "n_train": len(tr), "n_val": len(va),
        "frozen": m_f, "ft": m_t, "delta_f1": m_t["f1_macro"] - m_f["f1_macro"], "delta_f1_ci": dci,
        "epoch_times_frozen": t1.times, "epoch_times_ft": t2.times,
        "trainable_frozen": p_frozen, "trainable_ft": p_ft, "transitions": trans,
        "val_errors_frozen": int((~cf).sum()), "top_conf_frozen": top_confusions(cms[0], classes),
        "top_conf_ft": top_confusions(cms[1], classes), "cam_center_frozen": ce_f, "cam_center_ft": ce_t,
        "history": {k: h1.history[k] + h2.history[k] for k in h1.history},
    })
    with open(RESULTS_PATH, "w") as f:
        json.dump(RESULTS, f, indent=1)
    del model, base, head, train_ds, val_ds
    keras.backend.clear_session(); gc.collect()

## 3.1 — Piloto: BRACOL

In [ ]:
run_if("BRACOL", run_pilot)

## 3.2 — Piloto: PBC

In [ ]:
run_if("PBC", run_pilot)

## 3.3 — Piloto: DeepWeeds

In [ ]:
run_if("DeepWeeds", run_pilot)

**📝 Interpretação (preencher após rodar):**
- **Curvas:** a validação acompanha o treino? Um *gap* crescente indica overfitting; loss de validação ainda caindo na 5ª época indica que a base congelada ainda não saturou.
- **Base congelada:** perto de 99% de F1 macro significa pouco espaço para mostrar ganho de fine-tuning e poucos erros para a análise visual. Valores muito baixos indicam um problema difícil, ou rótulos ruidosos.
- **Margem do FT:** o ganho de F1 macro em 3 épocas é maior que o IC pareado? Se o IC inclui zero, o ganho não é distinguível de ruído *neste piloto* (no trabalho final, com mais épocas, pode ser diferente).
- **Confusões:** as principais confusões fazem sentido biológico ou agronômico (ex.: `ig` × `neutrophil`; ferrugem × cercosporiose)? Confusões interpretáveis = boa matéria-prima para a análise.

---
# Etapa 4 — Explicabilidade rápida (Grad-CAM)

**Decisões:**
- **Camada investigada:** a saída da última camada convolucional da base (MobileNetV2: `out_relu`; ResNet50: `conv5_block3_out`), um mapa de 7×7 que é o padrão para Grad-CAM nessas redes. A resolução é grosseira: cada célula do mapa cobre ~32×32 pixels.
- **Classe investigada:** a classe **prevista** por cada modelo (indicada no título). O gradiente é calculado sobre o *logit* (antes do softmax), para não saturar quando a confiança é ~100%. Com a cabeça GAP + Dense, o Grad-CAM coincide com o CAM clássico.
- **Imagens:** 2 acertos aleatórios e os 2 erros mais confiantes da **base congelada** (de classes reais distintas), mostrados **nas duas condições** (congelada e FT), como a lauda pede para comparar condições nas mesmas imagens.
- **Energia central:** a fração da "massa" do mapa nos 25% centrais da imagem (o valor de um mapa uniforme é 0,25). É só um indicador auxiliar, mais útil no PBC (célula centralizada). **O julgamento "objeto × fundo" é visual.**

In [ ]:
def show_gradcam(name):
    if name not in CAMS:
        print(f"{name}: rode antes o piloto (Etapa 3)."); return
    C = CAMS[name]; n = len(C["imgs"]); cl = C["classes"]
    fig, axes = plt.subplots(n, 3, figsize=(10.5, 3.5 * n))
    axes = np.atleast_2d(axes)
    for r in range(n):
        img, yt = C["imgs"][r], int(C["y"][r])
        pf, pt = int(C["prob_f"][r].argmax()), int(C["prob_t"][r].argmax())
        status = {(True, True): "acerto mantido", (False, True): "erro CORRIGIDO",
                  (False, False): "erro PERSISTENTE", (True, False): "erro NOVO"}[(pf == yt, pt == yt)]
        axes[r, 0].imshow(img)
        axes[r, 0].set_title(f"[{C['kinds'][r]}] real: {cl[yt]}\n{status}", fontsize=8)
        for c, (tag, prob, cam) in enumerate([("congelada", C["prob_f"][r], C["cams_f"][r]),
                                              ("FT", C["prob_t"][r], C["cams_t"][r])], start=1):
            ov, cam_r = overlay(img, cam); p = int(prob.argmax())
            axes[r, c].imshow(ov)
            axes[r, c].set_title(f"{tag}: previsto {cl[p]} ({prob[p]:.2f})\nclasse investigada: {cl[p]} | "
                                 f"energia central {center_energy(cam_r):.2f}", fontsize=8)
        for a in axes[r]: a.axis("off")
    fig.suptitle(f"{name} — Grad-CAM, camada '{C['layer']}' ({BACKBONE}); validação", fontsize=10)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / f"{name}_gradcam.png", dpi=110); plt.show()

for name in ["BRACOL", "PBC", "DeepWeeds"]:
    if name in SCANS:
        show_gradcam(name)

**📝 Interpretação (preencher após rodar), por dataset:** o destaque cai **no objeto de interesse** (a lesão na folha; o núcleo e o citoplasma da célula; a planta daninha) ou **no fundo** (hemácias ao redor, solo, céu, bordas)? Nos erros, o mapa aponta para uma região ambígua (ex.: duas lesões diferentes no mesmo recorte; duas células na imagem)? O FT desloca o foco? Mapas no fundo + teste de atalho positivo (Etapa 2) = forte indício de atalho.

---
# Etapa 5 — Comparação final

**Como as notas são atribuídas (transparência para a defesa):**
- **Calculadas a partir dos números deste notebook:** (c) vazamento/duplicatas, (d) espaço para análise, (e) margem do FT, (h) custo computacional, e a parte de contagens de (a).
- **Julgamento a priori, documentado na justificativa** (revise depois de olhar as saídas): (b) qualidade da fonte e dos rótulos, (f) potencial do Grad-CAM, (g) contexto, (i) imagem externa e o grau de "amplamente documentado" de (a). Nos critérios (b) e (f) há ajustes automáticos: ruído de rótulo detectado reduz (b), e menos de 10 erros na validação reduz (f).
- **Pesos:** mais peso para (a), por ser eliminatório, e para (d), porque a nota valoriza a análise. Os pesos ficam no dicionário `WEIGHTS` e podem ser alterados.

In [ ]:
WEIGHTS = {"a": 2.0, "b": 1.5, "c": 1.5, "d": 2.0, "e": 1.5, "f": 1.5, "g": 1.0, "h": 1.0, "i": 0.5}
CRITERIA = {
    "a": "Conformidade com a lauda", "b": "Qualidade/confiabilidade dos rótulos e da fonte",
    "c": "Risco de vazamento / duplicatas", "d": "Espaço para análise (nem trivial, nem impossível)",
    "e": "Margem de ganho com fine-tuning", "f": "Potencial do Grad-CAM e da análise visual de erros",
    "g": "Clareza do contexto e da finalidade prática", "h": "Custo computacional no Colab gratuito",
    "i": "Facilidade de achar imagem externa",
}
A_PRIORI = {
    "BRACOL": {
        "doc": (0, "pouco documentado (artigo + GitHub dos autores) — confirme com uma busca no Kaggle e no Papers with Code"),
        "b": (3, "rótulos dos autores (artigo em Comput. Electron. Agric. 2020), mas o zip do DOI está truncado, a versão "
                 "'symptom' só existe no GitHub e ~22% das imagens vêm do Digipathos/Embrapa (nenhuma saudável)"),
        "f": (4, "lesões visíveis e localizadas no recorte: dá para conferir se o mapa cai na lesão"),
        "g": (5, "café arábica no Brasil: confundir ferrugem com cercosporiose muda o manejo; um falso 'saudável' atrasa o controle"),
        "i": (5, "fotos de ferrugem e bicho-mineiro em folha de café são abundantes (Embrapa, extensão rural)"),
    },
    "PBC": {
        "doc": (1, "muito usado (cópias no Kaggle e dezenas de artigos) — risco real de ser visto como 'amplamente documentado'"),
        "b": (5, "Data in Brief 2020; CellaVision DM96 (Hospital Clínic de Barcelona); rotulado por patologistas clínicos"),
        "f": (4, "célula centralizada: fácil verificar se o mapa cai na célula ou nas hemácias/fundo"),
        "g": (5, "triagem hematológica: deixar passar granulócitos imaturos ou eritroblastos (anormais no sangue periférico) é o erro mais grave"),
        "i": (4, "há esfregaços na internet (ex.: Wikimedia Commons), mas com coloração e ampliação diferentes → domain shift"),
    },
    "DeepWeeds": {
        "doc": (2, "está no catálogo do TensorFlow Datasets, com tutoriais e benchmarks públicos — alto risco de veto"),
        "b": (4, "Scientific Reports 2019, imagens de campo rotuladas por especialistas; classe 'Negative' heterogênea"),
        "f": (3, "cenas de campo com fundo complexo e alvo pouco delimitado: mapas difíceis de julgar"),
        "g": (4, "controle de daninhas: falso negativo deixa a daninha; falso positivo aplica herbicida em planta nativa"),
        "i": (4, "há fotos das espécies, mas o enquadramento de campo do dataset é específico e 'Negative' não tem equivalente claro"),
    },
}
clip = lambda s: int(max(1, min(5, round(s))))

def score_dataset(name):
    S, Q, R, P = {}, QUALITY.get(name, {}), RESULTS.get(name), A_PRIORI[name]
    c = COMPLIANCE.get(name)
    if c:
        ok = c["cumpre ≥1.000 imagens"] and c["cumpre ≥100 por classe"]
        mn = SCANS[name]["df"].label.value_counts().min()
        base = (5 if mn >= 200 else 4) if ok else 1
        S["a"] = (clip(base - P["doc"][0]),
                  f"{c['imagens']} imagens, menor classe {c['menor classe']}, razão {c['razão maior/menor']}; {P['doc'][1]}")
    noise = Q.get("near_cross", 0) + Q.get("exact_cross", 0)
    S["b"] = (clip(P["b"][0] - (1 if Q and noise / max(Q["n"], 1) > 0.005 else 0)),
              P["b"][1] + (f"; {noise} pares duplicados com rótulos diferentes" if noise else ""))
    if Q:
        r = Q["frac_in_dup"]
        s = 5 if r < 0.005 else 4 if r < 0.02 else 3 if r < 0.05 else 2 if r < 0.15 else 1
        s -= 1 if Q["naive_groups_cross_test"] > 0 else 0
        s -= 1 if Q["frac_in_aug"] > 0.01 else 0
        txt = (f"{100 * r:.1f}% das imagens em quase-duplicatas; divisão ingênua poria {Q['naive_pairs_cross_test']} pares "
               f"duplicados e {Q['naive_groups_cross_test']} grupos de origem cruzando com o teste; "
               f"{100 * Q['frac_in_aug']:.1f}% em pares espelho/rotação")
        if Q.get("official_groups_cross"):
            txt += f"; split oficial: {Q['official_groups_cross']} grupos em >1 split"
        S["c"] = (clip(s), txt + " (mitigado pela divisão por grupos)")
    if R:
        f1 = R["frozen"]["f1_macro"]
        s = 5 if 0.70 <= f1 < 0.92 else 4 if 0.92 <= f1 < 0.96 else 3 if (0.96 <= f1 < 0.98 or 0.55 <= f1 < 0.70) else 2
        S["d"] = (s, f"F1 macro congelada = {f1:.3f}; {R['val_errors_frozen']} erros na validação; "
                     f"confusões: {'; '.join(R['top_conf_frozen'][:2])}")
        dlt, ci = R["delta_f1"], R["delta_f1_ci"]
        s = 5 if dlt >= 0.05 else 4 if dlt >= 0.03 else 3 if dlt >= 0.01 else 2 if dlt >= 0 else 1
        S["e"] = (s, f"Δ F1 macro em {EPOCHS_FT} épocas = {dlt:+.3f} (IC95% {ci[0]:+.3f} a {ci[1]:+.3f})"
                     + ("; IC inclui 0" if ci[0] <= 0 <= ci[1] else "") + f"; teto restante = {1 - R['ft']['f1_macro']:.3f}")
        s = P["f"][0] - (1 if R["val_errors_frozen"] < 10 else 0)
        S["f"] = (clip(s), P["f"][1] + f"; energia central média do CAM {np.mean(R['cam_center_frozen']):.2f} (uniforme = 0,25)")
        tot = sum(R["epoch_times_frozen"]) + sum(R["epoch_times_ft"])
        s = 5 if tot < 300 else 4 if tot < 600 else 3 if tot < 1200 else 2 if tot < 2400 else 1
        S["h"] = (s, f"piloto total {tot / 60:.1f} min ({R['n_train']} imagens de treino); o trabalho final (mais épocas, A e B) custa várias vezes isso")
    S["g"] = P["g"]; S["i"] = P["i"]
    return S

SCORES = {name: score_dataset(name) for name in DATASETS_TO_RUN if name in SCANS}
rows = []
for k, label in CRITERIA.items():
    row = {"critério": f"({k}) {label}", "peso": WEIGHTS[k]}
    for name, S in SCORES.items():
        row[name] = f"{S[k][0]} — {S[k][1]}" if k in S else "— (não calculado)"
    rows.append(row)
TABLE = pd.DataFrame(rows).set_index("critério")
with pd.option_context("display.max_colwidth", 400):
    display(TABLE.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

totals = {}
for name, S in SCORES.items():
    ks = [k for k in WEIGHTS if k in S]
    totals[name] = sum(WEIGHTS[k] * S[k][0] for k in ks) / sum(WEIGHTS[k] for k in ks)
summary = pd.DataFrame({
    "nota ponderada (1–5)": {n: round(v, 2) for n, v in totals.items()},
    "critérios calculados": {n: len(S) for n, S in SCORES.items()},
    "cumpre requisitos mínimos": {n: bool(COMPLIANCE[n]["cumpre ≥1.000 imagens"] and COMPLIANCE[n]["cumpre ≥100 por classe"]) for n in SCORES},
    "F1 macro val (congelada → FT)": {n: (f"{RESULTS[n]['frozen']['f1_macro']:.3f} → {RESULTS[n]['ft']['f1_macro']:.3f}" if n in RESULTS else "—") for n in SCORES},
}).sort_values("nota ponderada (1–5)", ascending=False)
display(summary)
with open(OUT_DIR / f"comparacao_final_{BACKBONE}.json", "w") as f:
    json.dump(to_py({"scores": {n: {k: list(v) for k, v in S.items()} for n, S in SCORES.items()},
                     "totals": totals, "weights": WEIGHTS, "quality": QUALITY}), f, indent=1, ensure_ascii=False)
TABLE.to_csv(OUT_DIR / f"comparacao_final_{BACKBONE}.csv")

In [ ]:
eligible = summary[summary["cumpre requisitos mínimos"]]
if len(eligible) == 0:
    print("Nenhum dataset elegível foi avaliado — verifique o carregamento.")
else:
    best = eligible.index[0]
    msg = f"### Recomendação pelos números: **{best}** (nota ponderada {eligible.iloc[0, 0]:.2f})\n"
    if len(eligible) > 1:
        gap = eligible.iloc[0, 0] - eligible.iloc[1, 0]
        second = eligible.index[1]
        msg += f"\nSegundo colocado: **{second}** ({eligible.iloc[1, 0]:.2f}); diferença de {gap:.2f}."
        if gap < 0.3:
            msg += ("\n\n⚠️ **Empate técnico** (diferença < 0,3). Desempate pelos critérios (a) documentação, "
                    "(b) fonte e (c) vazamento, e pela validação da proposta com a professora.")
    if summary["critérios calculados"].min() < len(WEIGHTS):
        msg += "\n\n⚠️ Alguns critérios não foram calculados (piloto incompleto para algum dataset); a nota usa só os disponíveis."
    display(Markdown(msg))

## 6. Leitura da recomendação e riscos para a defesa

### Minha expectativa *antes* de rodar (hipótese — confirme ou refute com a tabela acima)

**BRACOL *symptom* tende a ser o melhor encaixe na lauda**, desde que os números confirmem estes três pontos: (1) a base congelada fica claramente abaixo de ~95% de F1 macro, (2) o FT traz um ganho maior que o IC e (3) o teste de atalho de "fonte" não for gritante. Motivos:
- é o **menos documentado** dos três, o que reduz o risco de veto;
- tem **contexto brasileiro** claro, com erros de gravidade diferente (cercosporiose × ferrugem pedem manejos distintos; um falso "saudável" atrasa o controle);
- as **confusões são interpretáveis** (lesões parecidas) e há classes desbalanceadas na medida certa para justificar F1 macro;
- é **barato** no Colab: dá para rodar A e B com folga e repetir experimentos;
- **imagem externa** é fácil de achar.

**PBC** é a alternativa mais "limpa": os melhores rótulos, padronização de aquisição e confusões muito interpretáveis (`ig` com subtipos, bastonete × segmentado). Mas ele é popular, e a lauda veta datasets amplamente documentados. Se a professora aceitar, é uma escolha segura; só cuidado com um F1 congelado já muito alto.

**DeepWeeds** tem o maior risco de veto (está no TFDS, com tutoriais públicos), o maior custo e uma classe *Negative* que domina o dataset. Ele só seria a melhor escolha se os outros dois falharem.

> Se a tabela contradisser esta hipótese, **siga a tabela** e explique na defesa por que a expectativa inicial estava errada. Isso conta a favor ("qualidade das decisões").

### Riscos a antecipar na defesa

**Se escolherem BRACOL**
1. **"Por que não usaram o arquivo do DOI?"** Mostrem a auditoria: o zip do Mendeley bate com o sha256, mas está truncado e não contém `symptom`. Citem o DOI **e** o GitHub dos autores **e** o artigo, e expliquem que a versão *symptom* só existe no GitHub. Validem isso com a professora antes.
2. **Fontes misturadas:** ~22% das imagens vêm do Digipathos (Embrapa), nenhuma delas saudável. Mostrem a acurácia por fonte e o teste de atalho, e discutam se o modelo pode estar aprendendo "estilo da foto" em vez de lesão. O Grad-CAM é a evidência para isso.
3. **Vazamento por folha:** vários recortes vêm da mesma folha, e o split oficial dos autores deixa folhas em mais de um split. Expliquem por que dividiram **por folha** (o resultado fica menor, porém honesto).
4. **Validação e teste pequenos** (~400 imagens cada; a classe saudável tem ~40 por split): reportem ICs e não interpretem diferenças de 1–2 pontos como ganho real.
5. **Recortes pequenos** (largura mínima ~76 px) ampliados para 224: pode haver borramento; mencionem isso como limitação.

**Se escolherem PBC**
1. **"Esse dataset não é amplamente documentado?"** Tenham a resposta pronta, ou a validação prévia da professora.
2. **Sem ID de paciente/lâmina:** a divisão é por imagem, então células do mesmo paciente podem estar em treino e teste. Declarem a limitação.
3. **Classe `ig` heterogênea** (MY, MMY, PMY): confusões com `neutrophil` e `monocyte` refletem a granularidade do rótulo, não só erro do modelo. Usem o subtipo na análise.
4. **Desempenho alto:** com pouca margem, o foco da nota vai para a análise de erros e o Grad-CAM; deixem isso explícito.

**Se escolherem DeepWeeds**
1. Risco de veto por estar no TFDS. 2. A classe *Negative* (~52%) torna a acurácia enganosa: usem F1 macro e *balanced accuracy*. 3. Rajadas e sessões de captura vazam entre splits se a divisão não for por grupo. 4. Viés de local de coleta. 5. Custo: cada experimento leva bem mais tempo.

**Para qualquer escolha**
- Este piloto **não otimiza nada**: os números servem só para comparar datasets, não são o resultado do trabalho.
- **Reutilizem o CSV de divisão salvo** (`splits/<dataset>_split_seed42.csv`). O teste foi separado aqui e nunca carregado: mostrem o `assert` se perguntarem.
- Escolher o dataset olhando a **validação** é legítimo; olhar o **teste** não seria.
- Rodem o piloto com a **mesma arquitetura que a equipe inscreveu** (`BACKBONE`); as notas podem mudar um pouco.
- Grad-CAM em 7×7 é grosseiro: falem em "região", não em "pixel".
- Na GPU há uma pequena não-determinação residual mesmo com seeds fixas; os ICs cobrem essa variação.